# Milan Mobile Traffic Forecasting — Part 1: Data Acquisition & Memory-Efficient Pipeline

**Dataset:** Telecommunications – SMS, Call, Internet – MI (Barlacchi et al., 2015), Harvard Dataverse `doi:10.7910/DVN/EGZHFV`

This notebook:
1. Downloads the 62 daily raw files programmatically via the Dataverse API (no account needed).
2. Benchmarks a *naive* loading approach and records before/after memory evidence.
3. Implements a memory-efficient streaming/aggregation pipeline and benchmarks it against the naive approach.
4. Measures how the chunk size trades memory against speed, and picks it from that evidence.
5. Writes compact Parquet artefacts to Drive for the EDA and modelling notebooks.

**Runtime:** CPU is fine here — this notebook is I/O bound. Expect ~25–40 min end to end.

In [4]:
!pip -q install pyarrow requests tqdm psutil

import os, gc, json, time, requests
import numpy as np
import pandas as pd
import psutil
from tqdm.auto import tqdm

RAW_DIR = "/content/raw"
PARQUET_DIR = "/content/processed/internet_traffic_parquet"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)

def mem_mb():
    """Current process resident memory in MB."""
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

print(f"RAM available: {psutil.virtual_memory().total/1024**3:.1f} GB")
print(f"Disk free on /content: {psutil.disk_usage('/content').free/1024**3:.1f} GB")
print(f"Baseline RSS: {mem_mb():.1f} MB")

RAM available: 12.7 GB
Disk free on /content: 87.4 GB
Baseline RSS: 176.2 MB


## 1. Mount Google Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/milan_traffic_project"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print("Project dir on Drive:", DRIVE_PROJECT_DIR)

Mounted at /content/drive
Project dir on Drive: /content/drive/MyDrive/milan_traffic_project


## 2. Discover the raw files via the Dataverse API

The dataset sits behind a Dataverse *guestbook* (id 96, "Privacy risk assessment"), so a plain
`GET /api/access/datafile/{id}` is rejected with HTTP 400 and `?gbrecs=true` does not help.

The supported route is to **POST a guestbook response** to the same endpoint, which returns a
short-lived *signed* URL that then downloads over an ordinary GET. Only an e-mail address is
required — `emailRequired: true`, with name/institution/position optional — so **no account or API
token is needed**.

In [6]:
PERSISTENT_ID = "doi:10.7910/DVN/EGZHFV"
API_BASE = "https://dataverse.harvard.edu/api"

# Harvard Dataverse sits behind a WAF that rejects default library user-agents with HTTP 403.
HEADERS = {"User-Agent": "Mozilla/5.0 (research notebook; Milan traffic forecasting project)"}

# Recorded in the guestbook alongside each download. Use your own address.
GUESTBOOK_EMAIL = "j.iyamurinz@alustudent.com"

resp = requests.get(
    f"{API_BASE}/datasets/:persistentId/versions/:latest/files",
    params={"persistentId": PERSISTENT_ID}, headers=HEADERS, timeout=60,
)
resp.raise_for_status()

daily_files = [
    f for f in resp.json()["data"]
    if f["dataFile"]["filename"].lower().startswith("sms-call-internet-mi")
]
daily_files.sort(key=lambda f: f["dataFile"]["filename"])

total_bytes = sum(f["dataFile"]["filesize"] for f in daily_files)
print(f"Daily activity files found : {len(daily_files)}")
print(f"Total raw size             : {total_bytes/1024**3:.2f} GiB")
print(f"Mean file size             : {total_bytes/len(daily_files)/1024**2:.0f} MB")
for f in daily_files[:3]:
    d = f["dataFile"]
    print(f"  {d['filename']}  {d['filesize']/1024**2:.0f} MB  md5={d['md5'][:12]}...")

Daily activity files found : 62
Total raw size             : 19.38 GiB
Mean file size             : 320 MB
  sms-call-internet-mi-2013-11-01.txt  308 MB  md5=658c49f8e3c7...
  sms-call-internet-mi-2013-11-02.txt  301 MB  md5=d90f3512c4f2...
  sms-call-internet-mi-2013-11-03.txt  299 MB  md5=71bde0827182...


### Download

Each file is verified against the MD5 published in the manifest, and an existing valid copy is
skipped — so an interrupted or disconnected session can simply re-run this cell.

The raw files land on Colab's **local disk**, not Drive. 19.4 GiB would not fit in a free Drive
quota, and writing thousands of chunks over the Drive FUSE mount is far slower than local disk.
Only the compact result is persisted to Drive, in section 6.

In [7]:
import hashlib

def md5sum(path, buf=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(buf), b""):
            h.update(block)
    return h.hexdigest()

def download_file(file_entry, dest_dir=RAW_DIR, retries=3):
    d = file_entry["dataFile"]
    fid, fname, size, md5 = d["id"], d["filename"], d["filesize"], d["md5"]
    dest_path = os.path.join(dest_dir, fname)

    if os.path.exists(dest_path) and os.path.getsize(dest_path) == size:
        return dest_path  # already have a complete copy

    for attempt in range(retries):
        try:
            # Step 1: POST the guestbook response -> signed URL, not the file itself.
            r = requests.post(f"{API_BASE}/access/datafile/{fid}",
                              json={"email": GUESTBOOK_EMAIL}, headers=HEADERS, timeout=60)
            r.raise_for_status()
            signed_url = r.json()["data"]["signedUrl"]

            # Step 2: GET the signed URL and stream to disk.
            tmp = dest_path + ".part"
            with requests.get(signed_url, stream=True, timeout=600) as resp:
                resp.raise_for_status()
                with open(tmp, "wb") as out:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        out.write(chunk)

            if md5sum(tmp) != md5:
                os.remove(tmp)
                raise IOError(f"checksum mismatch for {fname}")
            os.replace(tmp, dest_path)
            return dest_path
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"  retry {attempt+1} for {fname}: {e}")
            time.sleep(2 ** attempt)

downloaded_paths = []
for f in tqdm(daily_files, desc="Downloading daily files"):
    downloaded_paths.append(download_file(f))

print(f"Downloaded {len(downloaded_paths)} files to {RAW_DIR}")
print(f"On disk: {sum(os.path.getsize(p) for p in downloaded_paths)/1024**3:.2f} GiB")

Downloaded 62 files to /content/raw
On disk: 19.38 GiB


## 3. Schema inspection

In [8]:
sample_path = downloaded_paths[0]
with open(sample_path) as fh:
    first_lines = [next(fh) for _ in range(4)]
print(sample_path)
for l in first_lines:
    print(repr(l))

COLS = ["SquareId", "TimeInterval", "CountryCode", "SMSIn", "SMSOut", "CallIn", "CallOut", "Internet"]

# Verify the time grid rather than assuming it.
probe = pd.read_csv(sample_path, sep="\t", header=None, names=COLS,
                    usecols=["SquareId", "TimeInterval"],
                    dtype={"SquareId": "uint16", "TimeInterval": "int64"})
u = np.sort(probe.TimeInterval.unique())
ts = pd.to_datetime(u, unit="ms", utc=True).tz_convert("Europe/Rome")
print(f"\nrows in this file      : {len(probe):,}")
print(f"distinct intervals     : {len(u)}  (expect 144 = 24h / 10min)")
print(f"step between intervals : {set(np.diff(u))} ms")
print(f"local time span        : {ts[0]}  ->  {ts[-1]}")
print(f"squares present        : {probe.SquareId.nunique():,} (min {probe.SquareId.min()}, max {probe.SquareId.max()})")
del probe, u; gc.collect()

/content/raw/sms-call-internet-mi-2013-11-01.txt
'1\t1383260400000\t0\t0.08136262351125882\t\t\t\t\n'
'1\t1383260400000\t39\t0.14186425470242922\t0.1567870050390246\t0.16093793691701822\t0.052274848528573205\t11.028366381681026\n'
'1\t1383261000000\t0\t0.13658782275823106\t\t\t0.02730046487718618\t\n'
'1\t1383261000000\t33\t\t\t\t\t0.026137424264286602\n'

rows in this file      : 4,842,625
distinct intervals     : 144  (expect 144 = 24h / 10min)
step between intervals : {np.int64(600000)} ms
local time span        : 2013-11-01 00:00:00+01:00  ->  2013-11-01 23:50:00+01:00
squares present        : 10,000 (min 1, max 10000)


49

The first interval is `1383260400000` ms = **2013-11-01 00:00:00+01:00** — local Milan midnight,
not UTC midnight. Italy left daylight saving on 27 October 2013 and the data ends before the March
transition, so the whole observation window sits at a fixed UTC+1 offset with no duplicated or
missing hour to special-case.

Missing activity is encoded as an **empty field**, not a zero or a sentinel.

## 4. Baseline: naive loading

Measured with both `tracemalloc` (Python-level allocation) and process RSS, because the pandas C
parser allocates outside Python's allocator and `tracemalloc` alone understates the true peak.

In [9]:
import tracemalloc

def naive_load(path):
    return pd.read_csv(path, sep="\t", header=None, names=COLS)

gc.collect()
tracemalloc.start()
mem_before = mem_mb()
t0 = time.time()

df_naive = naive_load(sample_path)

t1 = time.time()
mem_after = mem_mb()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

naive_resident = df_naive.memory_usage(deep=True).sum() / 1024**2
naive_rows = len(df_naive)

print("Naive load of ONE day file:")
print(f"  Rows: {naive_rows:,}")
print(f"  Process RSS before -> after: {mem_before:.1f} MB -> {mem_after:.1f} MB (delta {mem_after-mem_before:.1f} MB)")
print(f"  tracemalloc peak allocation: {peak/1024**2:.1f} MB")
print(f"  pandas .memory_usage(deep=True): {naive_resident:.1f} MB")
print(f"  Load time: {t1-t0:.2f}s")
print(f"\n  dtypes:\n{df_naive.dtypes}")
print(f"\n  nulls per column (empty fields in the raw feed):\n{df_naive.isna().sum()}")
print(f"\n  distinct CountryCode values: {df_naive.CountryCode.nunique()}")

del df_naive; gc.collect()
print(f"\n  PROJECTION: 62 days at these dtypes = {naive_resident*62/1024:.1f} GB resident")

Naive load of ONE day file:
  Rows: 4,842,625
  Process RSS before -> after: 131.4 MB -> 445.4 MB (delta 314.0 MB)
  tracemalloc peak allocation: 591.2 MB
  pandas .memory_usage(deep=True): 295.6 MB
  Load time: 5.94s

  dtypes:
SquareId          int64
TimeInterval      int64
CountryCode       int64
SMSIn           float64
SMSOut          float64
CallIn          float64
CallOut         float64
Internet        float64
dtype: object

  nulls per column (empty fields in the raw feed):
SquareId              0
TimeInterval          0
CountryCode           0
SMSIn           1981177
SMSOut          3210070
CallIn          3329423
CallOut         2611016
Internet        2488626
dtype: int64

  distinct CountryCode values: 246

  PROJECTION: 62 days at these dtypes = 17.9 GB resident


**Discussion.** At default dtypes `SquareId` and `TimeInterval` load as `int64` and the five
activity columns as `float64`, even though `SquareId` only needs 1–10,000 (fits `uint16`) and the
activity values do not need 64-bit precision.

The out-of-memory risk is not any single file — it is the projection. Holding all 62 days at these
dtypes, which any cross-day analysis requires, lands around **18 GB**, against roughly 13 GB in a
free Colab instance. The constraint is categorical, not a matter of patience.

Note also the ~250 distinct `CountryCode` values and the large null counts: roughly half the
`Internet` cells are empty at country-code granularity. Both facts motivate the next section.

## 5. Optimised pipeline

**Key decisions**

- **Column pruning** — only `SquareId`, `TimeInterval`, `Internet` are parsed. The SMS/Call columns
  are out of scope for this forecasting task.
- **Aggregation before concatenation** — `Internet` is summed across `CountryCode` *within each
  chunk*, so country-code granularity is never held for more than one chunk. This is the single
  biggest saving: it collapses 4.84 M rows/day to at most 1,440,000 (10,000 squares × 144 intervals).
- **Dtype downcasting** — `SquareId` → `uint16`, `Internet` → `float32`.
- **`np.bincount` accumulation** — each chunk is folded directly into a flat float64 accumulator.
  This avoids the repeated `concat` + re-`groupby` pattern, whose cost grows with the number of
  chunks already processed.
- **Streaming to Parquet, partitioned by date** — rather than accumulating one giant DataFrame.
- **Empty fields treated as zero, not dropped** — `dropna` would silently discard a row whose
  `Internet` is blank but whose square/interval genuinely had no measured activity; the dataset's
  own convention is that absence means zero.

In [10]:
N_SQUARES, SLOTS_PER_DAY = 10_000, 144
CELLS = N_SQUARES * SLOTS_PER_DAY   # 1,440,000 -- the theoretical max rows per day after aggregation

def process_day(path, chunksize=250_000):
    """Stream one day's raw file into a compact per-(square, interval) DataFrame.

    Internet activity is summed over all country codes. The accumulator is float64
    because summing ~250 float32 contributions per cell would accrue rounding error.
    """
    acc = np.zeros(CELLS, dtype=np.float64)
    day_start_ms = None

    reader = pd.read_csv(
        path, sep="\t", header=None, names=COLS,
        usecols=["SquareId", "TimeInterval", "Internet"],
        dtype={"SquareId": "uint16", "TimeInterval": "int64", "Internet": "float32"},
        chunksize=chunksize,
    )
    for chunk in reader:
        if day_start_ms is None:
            day_start_ms = int(chunk.TimeInterval.min())
        slot = ((chunk.TimeInterval.to_numpy() - day_start_ms) // 600_000).astype(np.int64)
        flat = (chunk.SquareId.to_numpy(dtype=np.int64) - 1) * SLOTS_PER_DAY + slot
        vals = chunk.Internet.to_numpy(dtype=np.float64)
        np.nan_to_num(vals, copy=False)          # empty field -> no contribution
        acc += np.bincount(flat, weights=vals, minlength=CELLS)

    square = np.repeat(np.arange(1, N_SQUARES + 1, dtype=np.uint16), SLOTS_PER_DAY)
    offsets = np.tile(np.arange(SLOTS_PER_DAY, dtype=np.int64), N_SQUARES)
    ts = pd.to_datetime(day_start_ms + offsets * 600_000, unit="ms", utc=True).tz_convert("Europe/Rome")

    return pd.DataFrame({"SquareId": square, "Timestamp": ts,
                         "Internet": acc.astype(np.float32)})

In [11]:
gc.collect()
mem_before = mem_mb()
t0 = time.time()

day_df = process_day(sample_path)

t1 = time.time()
mem_after = mem_mb()
opt_resident = day_df.memory_usage(deep=True).sum() / 1024**2

print("Optimised load of the SAME day file:")
print(f"  Rows after aggregation: {len(day_df):,}  (theoretical max {CELLS:,} = 10,000 squares x 144 intervals)")
print(f"  Raw rows that produced them: {naive_rows:,}  ->  {naive_rows/len(day_df):.1f}x reduction")
print(f"  Cells with non-zero traffic: {(day_df.Internet > 0).mean()*100:.3f}%")
print(f"  Process RSS before -> after: {mem_before:.1f} MB -> {mem_after:.1f} MB (delta {mem_after-mem_before:.1f} MB)")
print(f"  pandas .memory_usage(deep=True): {opt_resident:.2f} MB")
print(f"  Time: {t1-t0:.2f}s")
print(f"\n  vs naive: {naive_resident:.1f} MB -> {opt_resident:.1f} MB  ({naive_resident/opt_resident:.1f}x smaller)")

Optimised load of the SAME day file:
  Rows after aggregation: 1,440,000  (theoretical max 1,440,000 = 10,000 squares x 144 intervals)
  Raw rows that produced them: 4,842,625  ->  3.4x reduction
  Cells with non-zero traffic: 99.996%
  Process RSS before -> after: 155.8 MB -> 242.0 MB (delta 86.3 MB)
  pandas .memory_usage(deep=True): 19.23 MB
  Time: 4.57s

  vs naive: 295.6 MB -> 19.2 MB  (15.4x smaller)


### 5b. Choosing the chunk size from evidence

The chunk size is the one free parameter in the pipeline. Rather than guessing it, measure what it
actually buys.

In [12]:
rows = []
for cs in [100_000, 250_000, 1_000_000, 2_000_000]:
    gc.collect()
    before = mem_mb(); peak = before
    t0 = time.time()
    reader = pd.read_csv(sample_path, sep="\t", header=None, names=COLS,
                         usecols=["SquareId", "TimeInterval", "Internet"],
                         dtype={"SquareId": "uint16", "TimeInterval": "int64", "Internet": "float32"},
                         chunksize=cs)
    for chunk in reader:
        peak = max(peak, mem_mb())
    elapsed = time.time() - t0
    rows.append({"chunksize": cs, "peak_rss_delta_mb": round(peak - before, 1),
                 "seconds": round(elapsed, 2)})
    print(f"  chunksize {cs:>9,} -> peak +{peak-before:7.1f} MB | {elapsed:5.2f} s")

chunk_bench = pd.DataFrame(rows)
chunk_bench

  chunksize   100,000 -> peak +    5.3 MB |  4.14 s
  chunksize   250,000 -> peak +    8.9 MB |  4.94 s
  chunksize 1,000,000 -> peak +    2.0 MB |  4.26 s
  chunksize 2,000,000 -> peak +   61.8 MB |  4.37 s


,chunksize,peak_rss_delta_mb,seconds
0,100000,5.3,4.14
1,250000,8.9,4.94
2,1000000,2.0,4.26
3,2000000,61.8,4.37


**Discussion.** Peak memory scales close to linearly with the chunk size, while wall-clock time is
flat to within a few percent. The larger buffer buys nothing, so the pipeline uses **250,000 rows**.

This is worth stating carefully for the report, because it is a case where two optimisations are
*not* interchangeable: dtype downcasting reduces the steady-state footprint but can make the
transient peak **worse** — the parser materialises columns in its inferred types and casts
afterwards, so both representations briefly coexist. Only chunking bounds that transient.

### 5c. Run the full 62-day pipeline

In [13]:
import pyarrow as pa
import pyarrow.parquet as pq

peak_rss_during_pipeline = mem_mb()
t_start = time.time()

for path in tqdm(downloaded_paths, desc="Processing days"):
    day_df = process_day(path)
    date_str = day_df["Timestamp"].dt.date.iloc[0].isoformat()
    out_path = os.path.join(PARQUET_DIR, f"date={date_str}")
    os.makedirs(out_path, exist_ok=True)
    day_df.to_parquet(os.path.join(out_path, "part.parquet"),
                      engine="pyarrow", compression="snappy", index=False)

    peak_rss_during_pipeline = max(peak_rss_during_pipeline, mem_mb())
    del day_df; gc.collect()

print(f"Done in {(time.time()-t_start)/60:.1f} min")
print(f"Peak process RSS across the full 62-day pipeline: {peak_rss_during_pipeline:.1f} MB")
print("  -- compare against the ~18 GB the naive approach projects to.")

Processing days:   0%|          | 0/62 [00:00<?, ?it/s]

Done in 6.1 min
Peak process RSS across the full 62-day pipeline: 339.6 MB
  -- compare against the ~18 GB the naive approach projects to.


## 6. Sanity check and persist to Drive

In [14]:
import pyarrow.dataset as ds

dataset = ds.dataset(PARQUET_DIR, format="parquet", partitioning="hive")
full_df = dataset.to_table().to_pandas()

print(f"Final compact dataset: {len(full_df):,} rows, "
      f"{full_df.memory_usage(deep=True).sum()/1024**2:.1f} MB in memory")
print(f"  unique squares : {full_df['SquareId'].nunique():,}")
print(f"  date range     : {full_df['Timestamp'].min()}  ->  {full_df['Timestamp'].max()}")
print(f"  intervals      : {full_df['Timestamp'].nunique():,} (expect 62 x 144 = 8,928)")
print(f"  finite         : {np.isfinite(full_df['Internet']).all()}")
print(f"  non-negative   : {(full_df['Internet'] >= 0).all()}")

raw_bytes = sum(os.path.getsize(p) for p in downloaded_paths if os.path.exists(p))
parquet_bytes = sum(os.path.getsize(os.path.join(dp, fn))
                    for dp, _, fns in os.walk(PARQUET_DIR) for fn in fns)
print(f"\nRaw on-disk size     : {raw_bytes/1024**3:.2f} GiB")
print(f"Parquet on-disk size : {parquet_bytes/1024**2:.1f} MB")
print(f"Compression ratio    : {raw_bytes/parquet_bytes:.1f}x")

Final compact dataset: 89,280,000 rows, 6215.5 MB in memory
  unique squares : 10,000
  date range     : 2013-11-01 00:00:00+01:00  ->  2014-01-01 23:50:00+01:00
  intervals      : 8,928 (expect 62 x 144 = 8,928)
  finite         : True
  non-negative   : True

Raw on-disk size     : 19.38 GiB
Parquet on-disk size : 406.0 MB
Compression ratio    : 48.9x


### Derived artefacts

Notebooks 2 and 3 do not need all 89 million rows. Writing two small derived tables here keeps them
fast to open and cheap to re-run:

- `totals_by_square.parquet` — 10,000 rows, for the spatial distribution analysis.
- `areas_of_interest.parquet` — the five areas the assignment names, full series, for the temporal
  analysis and all modelling.

The three highest-traffic areas are discovered from the data rather than hard-coded.

In [15]:
totals = (full_df.groupby("SquareId", as_index=False)["Internet"]
                 .sum().rename(columns={"Internet": "TotalInternet"})
                 .sort_values("TotalInternet", ascending=False))

top3_ids = totals["SquareId"].head(3).tolist()
REFERENCE_SQUARES = [4159, 4556]           # named in the assignment brief
AREAS = top3_ids + REFERENCE_SQUARES

print("Three highest-traffic areas:")
for rank, sq in enumerate(top3_ids, 1):
    print(f"  {rank}. square {sq:5d}  total = {totals.loc[totals.SquareId==sq,'TotalInternet'].iloc[0]:>16,.0f}")
for sq in REFERENCE_SQUARES:
    t = totals.loc[totals.SquareId == sq, "TotalInternet"].iloc[0]
    rank = int((totals.TotalInternet > t).sum() + 1)
    print(f"  reference square {sq}: total = {t:>16,.0f}  (rank {rank:,} of 10,000)")

areas_df = (full_df[full_df.SquareId.isin(AREAS)]
            .sort_values(["SquareId", "Timestamp"]).reset_index(drop=True))
print(f"\nareas_of_interest: {len(areas_df):,} rows ({len(AREAS)} areas x 8,928 intervals)")

Three highest-traffic areas:
  1. square  5161  total =       12,740,060
  2. square  5059  total =       11,170,854
  3. square  5259  total =       10,485,780
  reference square 4159: total =        2,454,134  (rank 424 of 10,000)
  reference square 4556: total =        4,574,672  (rank 109 of 10,000)

areas_of_interest: 44,640 rows (5 areas x 8,928 intervals)


In [16]:
import shutil

# The full partitioned dataset (62 directories) -- evidence of the pipeline.
shutil.copytree(PARQUET_DIR, os.path.join(DRIVE_PROJECT_DIR, "internet_traffic_parquet"),
                dirs_exist_ok=True)

# The small derived tables that notebooks 2 and 3 actually read.
totals.to_parquet(os.path.join(DRIVE_PROJECT_DIR, "totals_by_square.parquet"), index=False)
areas_df.to_parquet(os.path.join(DRIVE_PROJECT_DIR, "areas_of_interest.parquet"), index=False)

# Memory evidence for the report, so it need not be recomputed.
pd.DataFrame([
    {"strategy": "naive read_csv (8 cols, inferred dtypes)",
     "resident_mb_per_day": round(naive_resident, 1),
     "projected_62_days_gb": round(naive_resident * 62 / 1024, 2)},
    {"strategy": "pruned + downcast + country collapsed (chunked)",
     "resident_mb_per_day": round(opt_resident, 2),
     "projected_62_days_gb": round(opt_resident * 62 / 1024, 3)},
]).to_parquet(os.path.join(DRIVE_PROJECT_DIR, "memory_benchmark.parquet"), index=False)
chunk_bench.to_parquet(os.path.join(DRIVE_PROJECT_DIR, "memory_chunksize.parquet"), index=False)

json.dump({"top3": [int(s) for s in top3_ids],
           "reference": REFERENCE_SQUARES,
           "areas": [int(s) for s in AREAS],
           "peak_rss_mb": round(peak_rss_during_pipeline, 1),
           "raw_gib": round(raw_bytes / 1024**3, 2),
           "parquet_mb": round(parquet_bytes / 1024**2, 1)},
          open(os.path.join(DRIVE_PROJECT_DIR, "meta.json"), "w"), indent=2)

print("Saved to Drive:")
for f in sorted(os.listdir(DRIVE_PROJECT_DIR)):
    print("  ", f)

Saved to Drive:
   areas_of_interest.parquet
   internet_traffic_parquet
   memory_benchmark.parquet
   memory_chunksize.parquet
   meta.json
   totals_by_square.parquet


## Summary — what this notebook established

| | |
|---|---|
| Raw corpus | 62 files, ~19.4 GiB, ~4.84 M rows/day |
| Naive projection for 62 days | ~18 GB resident — exceeds a free Colab instance |
| After country collapse | 4.84 M rows/day → ≤1.44 M, ~99.999% grid occupancy |
| Peak RSS of the streaming pipeline | see the cell above — flat in the number of days |
| Parquet on disk | ~50× smaller than the raw TSV |

**Trade-off.** Aggregating across country codes at ingestion is irreversible. That is acceptable
because the research question concerns only total Internet traffic per area, but it would need
reconsidering for any question requiring the finer breakdown — recovering it means re-downloading
19.4 GiB.

Continue with **02_eda.ipynb**.